# W4C2 solution: how scary is it?

The code we wrote together at the front of the room, complete and in order, so
you have the method while you write the comedy version yourself.

**Run every cell from the top. Everything here already works.**

The method in one sentence: three hand-written lists of adjectives turn each
film's reviews into three counts, and four trainable numbers turn those counts
into a prediction.

Your job in the lab is the same seven stages on `comedy_reviews.csv`, predicting
`funniness` from `funny.txt`, `crude.txt` and `flat.txt`. **Do not copy this
notebook across.** Read the one stage you are stuck on, close it, and type the
comedy version yourself.

## Stage 1. Load the films

One row per film. The two columns that matter sit at opposite ends of the
problem: `reviews` is a string, `scariness` is a number out of ten. Nothing in
PyTorch can multiply a string, so the whole job is crossing that gap.

Expect `(120, 5)`.

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path("../exercise/data")

df = pd.read_csv(DATA / "horror_reviews.csv")
print(df.shape)
print(df.loc[0, "title"], "|", df.loc[0, "scariness"])
print(df.loc[0, "reviews"])

## Stage 2. Read the rule in

The three lists are the rule, and a person wrote them by hand. They ship as
plain text so you can open one, read it out and argue with it: there is a word
missing from `fear.txt`, and probably one that should not be there.

`.read_text().split()` on a file of one word per line gives a list of words.
That is the whole of the parsing.

Expect `8 5 5`.

In [ ]:
lists = {name: (DATA / f"{name}.txt").read_text().split()
         for name in ("fear", "gore", "dull")}

print(len(lists["fear"]), len(lists["gore"]), len(lists["dull"]))
print(lists["fear"])

## Stage 3. Count the words

This is **rule-based feature extraction**, and it is what people built before
anything was learned from data.

Note what the loop is over: the three *categories*, not the 120 films.
`.str.count` already reaches every row at once, which is why there is no loop
over rows anywhere in this notebook.

Note also what it throws away, which is nearly everything: word order, who said
it, and negation. "Not remotely terrifying" counts as one fear word. Keep that
objection; it is what the rest of the course is for.

Expect the first film, Red House, to come out `1  0  0`.

In [ ]:
low = df["reviews"].str.lower()
for name, words in lists.items():
    df[name] = sum(low.str.count(word) for word in words)

print(df[["title", "fear", "gore", "dull", "scariness"]].head(3).to_string(index=False))

## Stage 4. Make them tensors

`float32` always. The error you get from the wrong dtype never contains the word
"dtype", which is why it costs everyone an afternoon once.

`X.shape` is `(120, 3)`: 120 films with three features each.

**The last line is the one people skip.** Centring subtracts each column's own
average so the average film sits at zero. Without it the fear weight lands in
about fifty steps and the bias is still crawling thousands of steps later,
because the bias has a much smaller gradient than a feature that averages 1.7.

In [ ]:
import torch

features = ["fear", "gore", "dull"]

X = torch.tensor(df[features].values, dtype=torch.float32)
y = torch.tensor(df["scariness"].values, dtype=torch.float32)
X = X - X.mean(0)

print(X.shape, X.dtype, "|", y.shape)

## Stage 5. The model is four numbers

No class, no `nn.Linear`, no framework. One number per word list, plus a height,
and you can hold the whole model in your head.

`requires_grad` is the switch: it tells PyTorch to record every operation these
take part in, and that recording is what makes them trainable. A tensor without
it is just an array.

In [ ]:
w = torch.zeros(len(features), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

print(w, b)

## Stage 6. One loss, one backward

Three beats: **predict, score, blame.** `X @ w + b` predicts all 120 films at
once, `- y` gives 120 errors, `** 2` makes a big miss hurt far more than a small
one, and `.mean()` collapses the lot to a single number.

Then `backward()` walks that one number back to everything with `requires_grad`
on and leaves a slope in `.grad`.

From all zeros: loss `32.041`, `w.grad` `[-4.113, -0.392, 1.125]`, `b.grad`
`-10.733`. Every gradient is negative except the dull one. Work out what that
says about which way each number has to move before you read on.

**Nothing has moved yet.** `backward()` fills in `.grad` and updates nothing.

In [ ]:
loss = ((X @ w + b - y) ** 2).mean()
loss.backward()

print(round(loss.item(), 3), w.grad, b.grad)

## Stage 7. Measure, blame, step, clear

`no_grad` because moving a parameter is not part of the model and must not be
recorded. `zero_()` because gradients **accumulate by default**, so a loop
without it trains on a running total. Everyone writes that bug exactly once, and
the two `zero_()` calls *above* the loop are that same bug caught one line
early: the cell above already ran a `backward()`, and nothing has cleared it.

After 200 steps at `lr = 0.05`, read the three weights out as three sentences:

- **fear +1.283**: a fear word is worth about a point and a quarter of scariness.
- **gore +0.334**: a quarter of that. Blood is not fear, and nobody told the
  model that; it found out by itself.
- **dull -0.587**: a dull word takes scariness *away*. That negative weight is
  the clearest evidence the model is reading words rather than just counting
  them.

In [ ]:
w.grad.zero_()
b.grad.zero_()

for step in range(200):
    loss = ((X @ w + b - y) ** 2).mean()      # measure
    loss.backward()                           # blame
    with torch.no_grad():                     # step
        w -= 0.05 * w.grad
        b -= 0.05 * b.grad
    w.grad.zero_()                            # clear
    b.grad.zero_()

print("fear %.3f   gore %.3f   dull %.3f   bias %.3f"
      % (*w.tolist(), b.item()))
print("loss %.4f" % loss.item())

## The check that matters

Loss `0.2055`, against **`3.2402`** for ignoring the reviews entirely and
guessing the average every time.

Always compute that second number. A model always returns an answer, and a
weight always looks like a finding. The do-nothing baseline is how you ask
whether either is worth anything.

In [ ]:
baseline = float(((y - y.mean()) ** 2).mean())

print("our model %.4f   guessing the average %.4f" % (loss.item(), baseline))

## Doing it on the comedies

Same seven stages on `comedy_reviews.csv`, predicting `funniness`. Two things
will be different, and neither is a bug:

1. **The loss will be much higher**, and that is the data, not your code. The
   comedy ratings are noisier because comedy is more divisive. Judge the model
   against its own baseline, never against the horror number.
2. **The weights will be slightly less exact.** Noise costs precision, not
   correctness, which is the honest reason to trust the method at all.

The task and the numbers to expect are in `../exercise/README.md`.